## Demo notebook

In this demo we will go through the basic functionalities of `RotOptSynth`.

In [1]:
import numpy as np
from scipy.stats import unitary_group
import rotoptsynth as ros
import pennylane as qml

In [2]:
# Pick some system size and generate a target with that size.
n = 4
n = 7
N = 2**n
wires = range(n)
dim = 4**n
U = unitary_group.rvs(N, random_state=81512)

In [ ]:
# Perform unitary synthesis with validation enabled.
ros.disable_validation()
%prun -s tottime ops = ros.rot_opt_qsd(U, wires=wires, zeroed_wires=None)

# Additionally check manually that the matrix is reproduced correctly:
print(f"Matrix reproduced correctly: {np.allclose(ros.ops_to_mat(ops, wires), U)}")

In [ ]:
# Count the number of rotation angles in the operators and compare to the group dimension
num_rotation_angles = ros.count_rotation_angles(ops)
print(f"Number of rotation angles ({num_rotation_angles}) matches group dimension ({dim}): {num_rotation_angles==dim}")

In [ ]:
# Draw the circuit
print(qml.drawer.tape_text(qml.tape.QuantumScript(ops), show_matrices=False, max_length=100, wire_order=wires))

In [ ]:
# Count gates:
@qml.specs
@qml.qnode(qml.device("default.qubit"))
def node():
    for op in ops:
        qml.apply(op)

    return qml.expval(qml.Z(0))
    
resources = node()["resources"]
gate_counts = dict(resources.gate_types)
gate_counts

In [ ]:
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(ros.asymmetric_two_qubit_decomp, show_matrices=False)(U, wires=wires))

In [ ]:
from rotoptsynth.flag_decomp import _flag_decomp_two_qubits
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(_flag_decomp_two_qubits, show_matrices=False)(U, wires=wires))

In [ ]:
ros.attach_multiplexer_node([qml.RY(0.6, 0)], [qml.RY(-0.1, 0)], "mpx")

In [ ]:
ops0 = [qml.SelectPauliRot([0.6, 0.7], [0], target_wire="target", rot_axis="X")]
ops1 = [qml.SelectPauliRot([0.2, -0.5], [0], target_wire="target", rot_axis="X")]
ros.attach_multiplexer_node(ops0, ops1, "new mpx")

In [ ]:
ros.attach_multiplexer_node([qml.CNOT([0, 1])], [qml.CNOT([0, 1])], "mpx")